# Logistic Regression — Purged Temporal Validation

This notebook benchmarks a core customer/behavior model against an extended model that adds subscription economics and plan-movement features. Because the churn label looks forward approximately one month, a one-month purge gap is placed between training observations and the out-of-time test period.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve

DATA_PATH = Path("../data/churn_modeling_ready.csv")
MODEL_DIR = Path("../models")
OUTPUT_DIR = Path("../data")
MODEL_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", None)


In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=["observation_date"])

print("Shape:", df.shape)
print("Date range:", df["observation_date"].min(), "to", df["observation_date"].max())
print("Churn observations:", int(df["is_churn"].sum()))
print("Overall churn rate:", f"{df['is_churn'].mean():.2%}")


## Feature sets

The core model uses customer attributes, raw behavior, and engineered behavioral ratios. The extended model adds plan, MRR, billing commitment, discount, MRR movement, and recent upgrade/downsell indicators.


In [ ]:
core_categorical = ["channel", "country"]

core_numeric = [
    "customer_age",
    "account_tenure_days",
    "post_per_month",
    "newfriend_per_month",
    "like_per_month",
    "adview_per_month",
    "dislike_per_month",
    "unfriend_per_month",
    "message_per_month",
    "reply_per_month",
    "total_engagement",
    "negative_engagement",
    "total_activity",
    "dislike_rate",
    "unfriend_rate",
    "reply_rate",
    "negative_activity_share",
    "adview_share",
]

core_features = core_categorical + core_numeric

extended_categorical = ["channel", "country", "current_plan"]

extended_numeric = core_numeric + [
    "current_mrr",
    "bill_period_months",
    "discount",
    "mrr_change",
    "upgrade_flag",
    "downsell_flag",
    "upgrade_last_90d",
    "downsell_last_90d",
]

extended_features = extended_categorical + extended_numeric

print("Core predictors:", len(core_features))
print("Extended predictors:", len(extended_features))


## Purged out-of-time split

The target construction looks one month ahead. Training therefore ends one month before the test starts. Observations inside that gap are intentionally excluded so the training labels are fully matured before the evaluation period begins.


In [ ]:
TEST_START = pd.Timestamp("2020-04-11")
TRAIN_END = TEST_START - pd.DateOffset(months=1)

train_df = df[df["observation_date"] < TRAIN_END].copy()
embargo_df = df[
    (df["observation_date"] >= TRAIN_END)
    & (df["observation_date"] < TEST_START)
].copy()
test_df = df[df["observation_date"] >= TEST_START].copy()

y_train = train_df["is_churn"].astype(int)
y_test = test_df["is_churn"].astype(int)

print("TRAIN")
print("Rows:", len(train_df))
print("Dates:", train_df["observation_date"].min(), "to", train_df["observation_date"].max())
print("Churns:", int(y_train.sum()), "| rate:", f"{y_train.mean():.2%}")

print("\nPURGE / EMBARGO")
print("Rows:", len(embargo_df))
print("Dates:", embargo_df["observation_date"].min(), "to", embargo_df["observation_date"].max())

print("\nTEST")
print("Rows:", len(test_df))
print("Dates:", test_df["observation_date"].min(), "to", test_df["observation_date"].max())
print("Churns:", int(y_test.sum()), "| rate:", f"{y_test.mean():.2%}")


In [ ]:
def make_logistic_pipeline(categorical_features, numeric_features):
    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", StandardScaler(), numeric_features),
            ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ]
    )

    model = LogisticRegression(
        max_iter=5000,
        solver="lbfgs",
        C=1.0,
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )


def evaluate_model(name, y_true, probabilities):
    return {
        "model": name,
        "roc_auc": roc_auc_score(y_true, probabilities),
        "pr_auc": average_precision_score(y_true, probabilities),
    }


In [ ]:
core_pipeline = make_logistic_pipeline(core_categorical, core_numeric)
extended_pipeline = make_logistic_pipeline(extended_categorical, extended_numeric)

core_pipeline.fit(train_df[core_features], y_train)
extended_pipeline.fit(train_df[extended_features], y_train)

core_probability = core_pipeline.predict_proba(test_df[core_features])[:, 1]
extended_probability = extended_pipeline.predict_proba(test_df[extended_features])[:, 1]

core_results = evaluate_model(
    "Core Behavior Logistic Regression",
    y_test,
    core_probability,
)

extended_results = evaluate_model(
    "Extended Subscription Logistic Regression",
    y_test,
    extended_probability,
)

comparison = pd.DataFrame([core_results, extended_results])
display(comparison)
print("Random PR baseline:", round(y_test.mean(), 4))


In [ ]:
core_fpr, core_tpr, _ = roc_curve(y_test, core_probability)
ext_fpr, ext_tpr, _ = roc_curve(y_test, extended_probability)

plt.figure(figsize=(7, 5))
plt.plot(core_fpr, core_tpr, label=f"Core (AUC={core_results['roc_auc']:.3f})")
plt.plot(ext_fpr, ext_tpr, label=f"Extended (AUC={extended_results['roc_auc']:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Logistic Regression ROC Curve")
plt.legend()
plt.tight_layout()
plt.show()

core_precision, core_recall, _ = precision_recall_curve(y_test, core_probability)
ext_precision, ext_recall, _ = precision_recall_curve(y_test, extended_probability)

plt.figure(figsize=(7, 5))
plt.plot(core_recall, core_precision, label=f"Core (AP={core_results['pr_auc']:.3f})")
plt.plot(ext_recall, ext_precision, label=f"Extended (AP={extended_results['pr_auc']:.3f})")
plt.axhline(y=y_test.mean(), linestyle="--", label="Random baseline")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Logistic Regression Precision-Recall Curve")
plt.legend()
plt.tight_layout()
plt.show()


## Save reproducible outputs


In [ ]:
predictions = test_df[
    [
        "account_id",
        "observation_date",
        "is_churn",
        "current_plan",
        "current_mrr",
        "upgrade_last_90d",
        "downsell_last_90d",
    ]
].copy()

predictions["churn_probability"] = extended_probability
predictions["risk_rank"] = predictions["churn_probability"].rank(method="first", ascending=False)
predictions["risk_percentile"] = predictions["churn_probability"].rank(pct=True)
predictions["monthly_revenue_at_risk"] = (
    predictions["churn_probability"] * predictions["current_mrr"]
)

joblib.dump(
    extended_pipeline,
    MODEL_DIR / "logistic_regression_extended.pkl",
)

predictions.to_csv(
    OUTPUT_DIR / "logistic_regression_test_predictions.csv",
    index=False,
)

comparison.to_csv(
    OUTPUT_DIR / "logistic_regression_model_comparison.csv",
    index=False,
)

print("Saved logistic model, predictions, and comparison table.")
